# EDA 8 - Case loactor maps after eda 7



### EDA 7 companion figure — mechanism choropleth

Shades ALL London MSOAs by a Frame C mechanism classification that generalises the three case-study narratives, then overlays the seven
case MSOAs (black outline + label) as exemplars of their zones.
 
**Categories (mutually exclusive, from eda4 Frame C typology):**
- genuine cascade   
  - Typ_C_21 == Cascade-led  AND  Casc_Inflow_Share_21 >= 0.5
  - (affluent-inflow-led: the gentrification signature)
- exodus cascade    
  - Typ_C_21 == Cascade-led  AND  Casc_Inflow_Share_21 <  0.5
  - (outflow-driven: the COVID-era false-cascade signature)
- cascade->counter  
  - Typ_C_11 == Cascade-led  AND  Typ_C_21 == Counter-led
- other             
  - everything else (light grey fabric)
 
**Geo Harmonisation**
Camden 024/025 merge: E02000190 inherits the category of E02000189.
 
**Inputs:**  
london_msoa_2011.geojson  (any GeoJSON with MSOA11CD works)
eda4_results_for_phase3_20260626.csv

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D
from shapely.geometry import shape
from shapely.ops import unary_union

plt.rcParams['font.family'] = 'DejaVu Sans'

In [ ]:
from pyprojroot import here

ROOT       = here()
sys.path.insert(0, str(ROOT))

DATA_DIR    = ROOT / 'data'
OUT_DIR     = ROOT / 'outputs' / 'case_study'
BOUNDARIES  = DATA_DIR / 'london_msoa_2011.geojson'
EDA4        = ROOT / 'outputs'/'eda4_results_for_phase3_20260626.csv'
OUT_PNG     = OUT_DIR / 'fig_case_locator_choropleth.png'
OUT_PNG_2    = OUT_DIR / 'fig_mechanism_geography_2011_2021.png'

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
C_IN, C_EX, K_IN = '#c0392b', '#e8a7a0', '#6a51a3'
GREY = '#e9e7e2'

In [ ]:
CAT_STYLE = {
    'inflow-driven': dict(fc=C_IN,     alpha=0.95),
    'outflow-driven':  dict(fc=C_EX,     alpha=0.95),
    'flip':    dict(fc=K_IN,     alpha=0.80),
    'other':   dict(fc=GREY,     alpha=1.00),
}

In [ ]:
CASES = {
    'E02000191': 'Camden 026',
    'E02000873': 'Tower Hamlets 010',
    'E02000809': 'Southwark 003',
    'E02000561': 'Islington 008',
    'E02000957': 'Wandsworth 035',
    'E02000440': 'Harrow 008',
    'E02000461': 'Harrow 029',
}

In [ ]:
LABEL_OFF = {
    'Camden 026':        (-0.105,  0.014),
    'Tower Hamlets 010': ( 0.095,  0.022),
    'Southwark 003':     ( 0.085, -0.048),
    'Islington 008':     ( 0.085,  0.036),
    'Wandsworth 035':    (-0.035, -0.058),
    'Harrow 008':        ( 0.082,  0.030),
    'Harrow 029':        (-0.090, -0.030),
}

In [ ]:
INNER_LONDON_LADS = {
    'E09000007','E09000001','E09000011','E09000012','E09000013',
    'E09000014','E09000019','E09000020','E09000022','E09000023',
    'E09000025','E09000028','E09000030','E09000032','E09000033',
}

In [ ]:
# ── classify ────────────────────────────────────────────────────────────
e4 = pd.read_csv(EDA4)
 
INFLOW_MIN = 0.25   # sits in the empirical gap (0.16 -> 0.31) among frame-robust cascades
 
def classify(r):
    if r['Typ_C_21'] == 'Cascade-led':
        robust    = r['Typ_A_21'] == 'Cascade-led'          # cascade under London frame too
        inflowled = r['Casc_Inflow_Share_21'] >= INFLOW_MIN # arm not overwhelmingly outflow
        return 'inflow-driven' if (robust and inflowled) else 'outflow-driven'
    
    ########################################################
    ### we need a further split of outflow-driven here 
    ### to differentiate exodus (external dominated) ot not.
    ########################################################

    if r['Typ_C_11'] == 'Cascade-led' and r['Typ_C_21'] == 'Counter-led':
        return 'flip'
    return 'other'
 
e4['mech'] = e4.apply(classify, axis=1)
cat = dict(zip(e4['msoa11cd'], e4['mech']))
cat['E02000190'] = cat.get('E02000189', 'other')     # Camden 024/025 merge
counts = e4['mech'].value_counts()

In [ ]:
# ── geometry ────────────────────────────────────────────────────────────
gj = json.load(open(BOUNDARIES))
geoms, codes, lads = [], [], []
for ft in gj['features']:
    geoms.append(shape(ft['geometry']))
    codes.append(ft['properties']['MSOA11CD'])
    lads.append(ft['properties'].get('LAD11CD', ''))
 
# def rings(geom):
#     polys = geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]
#     return [np.asarray(p.exterior.coords) for p in polys]

def rings(geom):
    # If it's a standard Polygon, wrap it in a list
    if geom.geom_type == 'Polygon':
        polys = [geom]
        
    # If it's a MultiPolygon, extract the constituent polygons
    elif geom.geom_type == 'MultiPolygon':
        polys = geom.geoms
        
    # If it's a GeometryCollection, filter out lines/points and keep only Polygons
    elif geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if g.geom_type == 'Polygon']
        
    # Catch-all for Points, LineStrings, or empty geometries
    else:
        polys = []
        
    return [np.asarray(p.exterior.coords) for p in polys]

######

fig, ax = plt.subplots(figsize=(10, 8))
 
# 1. choropleth, drawn per category so legend colours match exactly
for c in ('other', 'outflow-driven', 'flip', 'inflow-driven'):          # genuine on top
    st = CAT_STYLE[c]
    polys = [r for g, code in zip(geoms, codes)
             if cat.get(code, 'other') == c for r in rings(g)]
    ax.add_collection(PolyCollection(polys, facecolor=st['fc'],
                                     edgecolor='white', linewidth=0.3,
                                     alpha=st['alpha'], zorder=2))
 
# 2. Greater London + Inner London outlines
london = unary_union(geoms)
inner  = unary_union([g for g, l in zip(geoms, lads) if l in INNER_LONDON_LADS])
for r in rings(london):
    ax.plot(r[:, 0], r[:, 1], color='#6f6f6f', lw=1.1, zorder=5)
for r in rings(inner):
    ax.plot(r[:, 0], r[:, 1], color='#3d3d3d', lw=1.1, ls=(0, (5, 3)), zorder=5)
 
# 3. case MSOAs: black outline + leader label (exemplars of their zone)
for code, name in CASES.items():
    g = geoms[codes.index(code)]
    ax.add_collection(PolyCollection(rings(g), facecolor='none',
                                     edgecolor='black', linewidth=1.5, zorder=6))
    cx, cy = g.centroid.x, g.centroid.y
    dx, dy = LABEL_OFF[name]
    ax.annotate(name, xy=(cx, cy), xytext=(cx + dx, cy + dy),
                fontsize=8, fontweight='bold', color='#111',
                ha='center', va='center', zorder=8,
                arrowprops=dict(arrowstyle='-', color='#111', lw=0.8,
                                shrinkA=0, shrinkB=1),
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='#999', lw=0.5, alpha=0.9))
 
# 4. legend with live counts
handles = [
    mpatches.Patch(fc=C_IN,  label=f"Genuine cascade — cascade-led in both frames, inflow-led (n={counts.get('inflow-driven',0)})"),
    mpatches.Patch(fc=C_EX,  label=f"Exodus cascade — 2021 cascade-led only, outflow-driven (n={counts.get('outflow-driven',0)})"),
    mpatches.Patch(fc=K_IN, alpha=0.8,
                   label=f"Cascade \u2192 counter flip, 2011\u21922021 (n={counts.get('flip',0)})"),
    mpatches.Patch(fc=GREY,  label=f"Other flow regimes (n={counts.get('other',0)})"),
    Line2D([0], [0], color='#3d3d3d', lw=1.1, ls=(0, (5, 3)), label='Inner London (statutory)'),
    Line2D([0], [0], color='black', lw=1.5, label='Case-study MSOA'),
]
ax.legend(handles=handles, loc='lower left', fontsize=7.4, frameon=False,
          borderaxespad=0.3)
 
ax.set_title('Three flow mechanisms across London \u2014 Frame C classification with case-study exemplars',
             fontsize=11.5, fontweight='bold', pad=10)
ax.set_aspect(1.0 / np.cos(np.radians(51.5)))
ax.set_xlim(-0.60, 0.36); ax.set_ylim(51.26, 51.72)
ax.set_axis_off()
 
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150, bbox_inches='tight')
print(f'saved {OUT_PNG}')
print(counts.to_string())

## Temporal flipping of counter-led

Same classification rules applied independently to each census year, side by side. Per-year categories (the 'flip' is a transition, so it
appears **BETWEEN the panels as counter-led purple flooding the inner core**):
 
- genuine cascade   
    - Typ_C_yr == Cascade-led AND Typ_A_yr == Cascade-led
    - AND Casc_Inflow_Share_yr >= 0.25
    - (frame-robust and inflow-led)
- exodus cascade    
    - Typ_C_yr == Cascade-led, otherwise
    - (frame-manufactured and/or outflow-driven)
- counter-led       
    - Typ_C_yr == Counter-led
- other             
    - Symmetric / Lateral (light grey)
 
Threshold 0.25 sits in an empirical gap of the inflow-share distribution among frame-robust cascades in BOTH years (2011: 0.23->0.30; 2021: 0.16->0.31).

In [ ]:
C_IN, C_EX, K_IN, GREY = '#c0392b', '#e8a7a0', '#6a51a3', '#e9e7e2'
CAT_STYLE = {'inflow-driven': dict(fc=C_IN, alpha=0.95),
             'outflow-driven':  dict(fc=C_EX, alpha=0.95),
             'counter': dict(fc=K_IN, alpha=0.60),
             'other':   dict(fc=GREY, alpha=1.00)}

In [ ]:
CASES = {
    'E02000191': 'Camden 026',        'E02000873': 'Tower Hamlets 010',
    'E02000809': 'Southwark 003',     'E02000561': 'Islington 008',
    'E02000957': 'Wandsworth 035',    'E02000440': 'Harrow 008',
    'E02000461': 'Harrow 029',
}

In [ ]:
LABEL_OFF = {
    'Camden 026':        (-0.115,  0.016),
    'Tower Hamlets 010': ( 0.105,  0.026),
    'Southwark 003':     ( 0.095, -0.052),
    'Islington 008':     ( 0.090,  0.040),
    'Wandsworth 035':    (-0.035, -0.062),
    'Harrow 008':        ( 0.088,  0.032),
    'Harrow 029':        (-0.095, -0.034),
}

In [ ]:
INNER_LONDON_LADS = {
    'E09000007','E09000001','E09000011','E09000012','E09000013',
    'E09000014','E09000019','E09000020','E09000022','E09000023',
    'E09000025','E09000028','E09000030','E09000032','E09000033',
}

In [ ]:
# ── classify per year ───────────────────────────────────────────────────
 
def classify(r, yr):
    if r[f'Typ_C_{yr}'] == 'Cascade-led':
        robust    = r[f'Typ_A_{yr}'] == 'Cascade-led'
        inflowled = r[f'Casc_Inflow_Share_{yr}'] >= INFLOW_MIN
        return 'inflow-driven' if (robust and inflowled) else 'outflow-driven'
    if r[f'Typ_C_{yr}'] == 'Counter-led':
        return 'counter'
    return 'other'
 
cat, counts = {}, {}
for yr in ('11', '21'):
    e4[f'mech_{yr}'] = e4.apply(classify, axis=1, yr=yr)
    d = dict(zip(e4['msoa11cd'], e4[f'mech_{yr}']))
    d['E02000190'] = d.get('E02000189', 'other')
    cat[yr] = d
    counts[yr] = e4[f'mech_{yr}'].value_counts()

In [ ]:
# ── geometry ────────────────────────────────────────────────────────────
gj = json.load(open(BOUNDARIES))
geoms, codes, lads = [], [], []
for ft in gj['features']:
    geoms.append(shape(ft['geometry']))
    codes.append(ft['properties']['MSOA11CD'])
    lads.append(ft['properties'].get('LAD11CD', ''))
 
def rings(geom):
    # If it's a standard Polygon, wrap it in a list
    if geom.geom_type == 'Polygon':
        polys = [geom]
        
    # If it's a MultiPolygon, extract the constituent polygons
    elif geom.geom_type == 'MultiPolygon':
        polys = geom.geoms
        
    # If it's a GeometryCollection, filter out lines/points and keep only Polygons
    elif geom.geom_type == 'GeometryCollection':
        polys = [g for g in geom.geoms if g.geom_type == 'Polygon']
        
    # Catch-all for Points, LineStrings, or empty geometries
    else:
        polys = []
        
    return [np.asarray(p.exterior.coords) for p in polys]

In [ ]:
# ── figure: two panels ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15.5, 7.2))
 
for ax, yr, label_cases in zip(axes, ('11', '21'), (False, True)):
    for c in ('other', 'counter', 'outflow-driven', 'inflow-driven'):
        st = CAT_STYLE[c]
        polys = [r for g, code in zip(geoms, codes)
                 if cat[yr].get(code, 'other') == c for r in rings(g)]
        if polys:
            ax.add_collection(PolyCollection(polys, facecolor=st['fc'],
                                             edgecolor='white', linewidth=0.25,
                                             alpha=st['alpha'], zorder=2))
    for r in rings(london):
        ax.plot(r[:, 0], r[:, 1], color='#6f6f6f', lw=1.0, zorder=5)
    for r in rings(inner):
        ax.plot(r[:, 0], r[:, 1], color='#3d3d3d', lw=1.0, ls=(0, (5, 3)), zorder=5)
 
    for code, name in CASES.items():
        g = geoms[codes.index(code)]
        ax.add_collection(PolyCollection(rings(g), facecolor='none',
                                         edgecolor='black', linewidth=1.4, zorder=6))
        if label_cases:
            cx, cy = g.centroid.x, g.centroid.y
            dx, dy = LABEL_OFF[name]
            ax.annotate(name, xy=(cx, cy), xytext=(cx + dx, cy + dy),
                        fontsize=7.4, fontweight='bold', color='#111',
                        ha='center', va='center', zorder=8,
                        arrowprops=dict(arrowstyle='-', color='#111', lw=0.7,
                                        shrinkA=0, shrinkB=1),
                        bbox=dict(boxstyle='round,pad=0.18', fc='white',
                                  ec='#999', lw=0.5, alpha=0.9))
 
    n = counts[yr]
    ax.set_title(f'20{yr}', fontsize=13, fontweight='bold', pad=6)
    ax.text(0.5, -0.015,
            f"inflow-driven {n.get('inflow-driven',0)}   \u00b7   outflow-driven {n.get('outflow-driven',0)}   \u00b7   "
            f"counter-led {n.get('counter',0)}   \u00b7   other {n.get('other',0)}",
            transform=ax.transAxes, ha='center', va='top', fontsize=9, color='#333')
    ax.set_aspect(1.0 / np.cos(np.radians(51.5)))
    ax.set_xlim(-0.60, 0.36); ax.set_ylim(51.26, 51.72)
    ax.set_axis_off()
 
handles = [
    mpatches.Patch(fc=C_IN, label='Genuine cascade \u2014 cascade-led in both frames, inflow-led'),
    mpatches.Patch(fc=C_EX, label='Exodus cascade \u2014 cascade-led via external flows / outflow-driven'),
    mpatches.Patch(fc=K_IN, alpha=0.6, label='Counter-led'),
    mpatches.Patch(fc=GREY, label='Symmetric / Lateral'),
    Line2D([0], [0], color='#3d3d3d', lw=1.0, ls=(0, (5, 3)), label='Inner London (statutory)'),
    Line2D([0], [0], color='black', lw=1.4, label='Case-study MSOA'),
]
fig.legend(handles=handles, loc='lower center', ncol=3, frameon=False,
           fontsize=8.6, bbox_to_anchor=(0.5, 0.005))
fig.suptitle('Flow-mechanism geography of London, 2011 vs 2021 \u2014 Frame C classification',
             fontsize=13.5, fontweight='bold', y=0.99)
 
plt.tight_layout(rect=[0, 0.075, 1, 0.965])
plt.savefig(OUT_PNG_2, dpi=150, bbox_inches='tight')
print(f'saved {OUT_PNG_2}')
for yr in ('11', '21'):
    print(f'20{yr}:', counts[yr].to_dict())

### Quick interpretation

Treat this map as the lead figure of the case-study section, showing the landscape and its changes. Then the alluvials from EDA_7 to dissect the 3 outlined mechanisms.

- Genuine cascades 
    - frame-robust, inflow-led, the closet to flow-defined gentrification
    - collapse from 56 to 13

- Exodus cascades
    - grow from 157 to 182
    - consolidate into the outer belt

- Counter-led
    - expands from 280 to 452
    - just compare colours, ***MSOAs that were red or grey in the inner core in 2011 are purple by 2021***
    - Southward, Islington, Wandsworth are worked examples

The 0.25 inflow-share threshold sits in an empirical gap in both years (0.23 to 0.3 in 2011, 0.16 to 0.31 in 2021), so the genuine and exodus split isn't threshold fragile.


> **We can map small multiple versions with the same pair but faceted by ring - inner and outer with the count table.**